In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline

df=pd.DataFrame({"size":[800,900,1000,1200,1400,1600,1800,2000],
                 "distance":[15,14,12,10,8,7,5,4],
                 "price":[35,38,42,50,58,65,75,82]})
print(df.head())
print(df.describe())

   size  distance  price
0   800        15     35
1   900        14     38
2  1000        12     42
3  1200        10     50
4  1400         8     58
              size   distance      price
count     8.000000   8.000000   8.000000
mean   1337.500000   9.375000  55.625000
std     437.321392   4.068608  17.394067
min     800.000000   4.000000  35.000000
25%     975.000000   6.500000  41.000000
50%    1300.000000   9.000000  54.000000
75%    1650.000000  12.500000  67.500000
max    2000.000000  15.000000  82.000000


In [12]:

X_train_s= df[["size", "distance"]]
y = df["price"]

# 2. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
scaler=StandardScaler()
X_train_s= scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 4 & 5. Train for K = 2, 3, 4, 5 and find the best K by R² score
scores = {}
for k in [2, 3, 4, 5]:
    model = KNeighborsRegressor(n_neighbors=k)
    model.fit(X_train_s, y_train)
    scores[k] = model.score(X_test_s, y_test)   # R² score
    print(f"K = {k}  ->  R² = {scores[k]:.4f}")

best_k = max(scores, key=scores.get)
print("Best K:", best_k)

# 6. Compare distance metrics (using the best K)
for metric, extra in [("euclidean", {}), ("manhattan", {}), ("minkowski", {"p": 3})]:
    model = KNeighborsRegressor(n_neighbors=best_k, metric=metric, **extra)
    model.fit(X_train_s, y_train)
    print(f"{metric:10s} -> R² = {model.score(X_test_s, y_test):.4f}")

# 7. Predict price for Size = 1500, Distance = 6
new_house = pd.DataFrame({"size": [1500], "distance": [6]})
new_scaled = scaler.transform(new_house)

final_model = KNeighborsRegressor(n_neighbors=best_k).fit(X_train_s, y_train)
print("Predicted price (Lakhs):", final_model.predict(new_scaled)[0])



K = 2  ->  R² = 0.9931
K = 3  ->  R² = 0.8266
K = 4  ->  R² = 0.8090
K = 5  ->  R² = 0.4267
Best K: 2
euclidean  -> R² = 0.9931
manhattan  -> R² = 0.9931
minkowski  -> R² = 0.9931
Predicted price (Lakhs): 66.5
